# 02 — Dictionary Baseline

**`BASE-01` — dictionary / position baseline** *(legacy id: the project's original baseline)*.

A non-neural lower bound: for each Arabic subword, predict the English subword most frequently
seen at the same *relative* position in training, then stitch the guesses together. It has no
learning and no context — it exists only to show how far a trivial lexical method gets, so the
trained Transformer (notebook 03) can be measured against it.

This notebook contains **only** the dictionary baseline — no Transformer results, no beam search.

## Setup

In [1]:
import os
from collections import Counter, defaultdict
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
import pandas as pd, sacrebleu, sentencepiece as spm
TOK, VOC, MAX_LEN = 'Data/tokenized', 'Data/vocab', 80
sp_ar = spm.SentencePieceProcessor(model_file=f'{VOC}/sp_ar.model')
sp_en = spm.SentencePieceProcessor(model_file=f'{VOC}/sp_en.model')
def read_lines(p): return open(p, encoding='utf-8').read().splitlines()

## Load data (same filtered test set as the Transformer)

In [2]:
train_ar, train_en = read_lines(f'{TOK}/train.ar.bpe'), read_lines(f'{TOK}/train.en.bpe')
kept = [(a, e) for a, e in zip(read_lines(f'{TOK}/test.ar.bpe'), read_lines(f'{TOK}/test.en.bpe'))
        if len(a.split()) <= MAX_LEN and len(e.split()) <= MAX_LEN]
test_ar = [a for a, e in kept]; test_en = [e for a, e in kept]
print(f'train {len(train_ar):,} | test kept {len(kept):,}')

train 50,000 | test kept 8,491


## The baseline algorithm
Most-frequent English subword per Arabic subword at the same relative position.

In [3]:
def build_position_dictionary(src_lines, tgt_lines):
    counts = defaultdict(Counter)
    for src, tgt in zip(src_lines, tgt_lines):
        s, t = src.split(), tgt.split()
        if not s or not t:
            continue
        for i, piece in enumerate(s):
            j = min(round(i * (len(t) - 1) / max(1, len(s) - 1)), len(t) - 1)
            counts[piece][t[j]] += 1
    return {p: c.most_common(1)[0][0] for p, c in counts.items()}

def translate_baseline(line, dictionary):
    return ' '.join(dictionary.get(p, '<unk>') for p in line.split())

dictionary = build_position_dictionary(train_ar, train_en)
print('dictionary entries:', f'{len(dictionary):,}')
list(dictionary.items())[:8]

dictionary entries: 8,173


[('▁اذاً', '▁so'),
 ('▁نحن', '▁we'),
 ('▁ن', '▁we'),
 ('بي', ','),
 ('عها', '▁to'),
 ('،', ','),
 ('▁ثم', '▁and'),
 ('▁شئ', '▁something')]

## Scoring (BLEU + chrF++ on detokenized English)

In [4]:
def detok(lines, sp): return [sp.decode(l.split()) for l in lines]
def score(hyp, ref):
    return (round(sacrebleu.corpus_bleu(hyp, [ref]).score, 4),
            round(sacrebleu.corpus_chrf(hyp, [ref], word_order=2).score, 4))

baseline = detok([translate_baseline(l, dictionary) for l in test_ar], sp_en)
reference = detok(test_en, sp_en)
bleu_full, chrf_full = score(baseline, reference)
bleu_1k, chrf_1k = score(baseline[:1000], reference[:1000])
results = pd.DataFrame([
    {'local_id': 'BASE-01', 'old_id': 'dictionary_position_baseline', 'split': 'test_full',
     'examples': len(baseline), 'bleu': bleu_full, 'chrf_pp': chrf_full},
    {'local_id': 'BASE-01', 'old_id': 'dictionary_position_baseline', 'split': 'test_1000',
     'examples': 1000, 'bleu': bleu_1k, 'chrf_pp': chrf_1k},
])
results

,local_id,old_id,split,examples,bleu,chrf_pp
0,BASE-01,dictionary_position_baseline,test_full,8491,4.7777,25.6503
1,BASE-01,dictionary_position_baseline,test_1000,1000,4.7137,25.5561


In [5]:
os.makedirs('outputs/tables', exist_ok=True)
pd.DataFrame([{'model': 'dictionary_position_baseline', 'local_id': 'BASE-01', 'decoding': 'position_dictionary',
               'examples': len(baseline), 'bleu': bleu_full, 'chrf_pp': chrf_full}]).to_csv(
    'outputs/tables/baseline_results.csv', index=False)
print('saved outputs/tables/baseline_results.csv')

saved outputs/tables/baseline_results.csv


## Qualitative examples
Word-salad: correct vocabulary, no grammar — exactly what a floor should look like.

In [6]:
samples = pd.DataFrame({'source_ar': detok(test_ar[:6], sp_ar), 'reference_en': reference[:6],
                        'baseline_output': baseline[:6]})
os.makedirs('outputs/examples', exist_ok=True)
samples.to_csv('outputs/examples/baseline_samples.csv', index=False, encoding='utf-8')
samples

,source_ar,reference_en,baseline_output
0,قبل عدة سنوات، هنا في تيد، قدّم بيتر سكيلمان م...,"several years ago here at ted, peter skillman ...","before many years, here in ted,,, peter, form,..."
1,والفكرة غاية في البساطة. فريق مكوّن من اربعة ي...,and the idea's pretty simple: teams of four ha...,"and the very in simplicity. a, a of four we to..."
2,يجب ان تكون المارش مالو علي القمة.,the marshmallow has to be on top.,"we to be the the, money, on the."
3,ورغماً عن انها تبدو بسيطة للغاية، الا انها صعب...,"and, though it seems really simple, it's actua...","and the, about it look simple., the it difficu..."
4,لذا فقد فكرت بان هذه فكرة مثيرة، وقمت بتضمينها...,"and so, i thought this was an interesting idea...","so we i that this idea interesting, and to,. i..."
5,وقد كان نجاحاً باهراً.,and it was a huge success.,"and was success, the,."


## Limitations of the dictionary baseline
No context, no reordering, no fluency — it cannot model agreement, word order, or multi-word
expressions. Its BLEU is very low; interestingly its **chrF++** is competitive (it emits many
correct content subwords). It is a lower bound only — the neural model is in notebook 03.